# Lojas virtuais do Brasil: exploratory analysis

63,974 Brazilian online stores (April 2023) with platform, SSL and Reclame Aqui; the 2022 e-commerce census; and 63,933 synthetic pages written by GPT-3.5. Dataset: [Kaggle](https://www.kaggle.com/datasets/caiquedourado/lojas-virtuais-brasil) · [GitHub](https://github.com/CaiqueDourado/lojas-virtuais-brasil) (methodology, data quality report, data dictionary).

Runs on Kaggle with the full files, or locally with the repository's `sample/`.

In [ ]:
from pathlib import Path
import re
import pandas as pd

KAGGLE = next((p.parent for p in Path("/kaggle/input").glob("**/lojas.parquet")), Path("/kaggle/input/none"))
def load(name):
    if KAGGLE.exists():
        return pd.read_parquet(KAGGLE / f"{name}.parquet")
    return pd.read_csv(Path("../sample") / f"{name}_sample.csv")

lojas, ra, gpt, censo = load("lojas"), load("reclame_aqui"), load("textos_gpt"), load("censo_2022")
print(f"{len(lojas):,} stores, {len(ra):,} Reclame Aqui profiles, {len(gpt):,} GPT pages, {len(censo):,} census domains")

## Platforms

In [ ]:
ax = lojas.plataforma.value_counts().head(15).sort_values().plot.barh(figsize=(8, 5), title="Stores per platform")
lojas.plataforma.isna().mean().round(3)

### Three tests that disagree

The declared platform against the names found in the HTML.

In [ ]:
pd.crosstab(lojas.plataforma, lojas.sinais_no_html.str.split(", ").str[0]).loc[
    ["WooCommerce", "Loja Integrada", "NuvemShop", "Shopify", "Wix"]].T.sort_values("WooCommerce", ascending=False).head(10)

## SSL certificates

In [ ]:
lojas.certificado_emissor.value_counts().head(8)

## Reclame Aqui

Most linked stores are NO_INDEX (too few complaints to be rated).

In [ ]:
lojas.reclame_aqui_status.value_counts(dropna=False)

In [ ]:
rated = lojas.dropna(subset=["reclame_aqui_nota"])
rated.groupby("plataforma").reclame_aqui_nota.agg(["count", "mean"]).query("count >= 10").sort_values("mean").round(2)

## What GPT-3.5 said

Each page has a section "A loja X é confiável?". How did the model answer, and does it depend on the store's real reputation?

In [ ]:
ANSWER = re.compile(r"conf[ií]?[aá]vel\??\s*</h2>\s*(.{0,120})", re.I | re.S)
def answer(text):
    m = ANSWER.search(text or "")
    if not m:
        return "no section"
    a = m.group(1).strip().lower()
    return "no" if a.startswith(("não", "nao")) else "yes"

gpt["answer"] = gpt.texto.map(answer)
gpt.answer.value_counts()

In [ ]:
j = gpt.merge(lojas[["id_loja", "reclame_aqui_status"]], on="id_loja")
pd.crosstab(j.reclame_aqui_status, j.answer)

In [ ]:
invented = gpt.texto.str.contains(r"fundad[ao] em \d{4}|desde \d{4}|há mais de \d+ anos", regex=True)
print(f"pages with a founding year or age: {invented.mean():.0%}")
print(gpt.texto.iloc[0][:800])

## The 2022 census

In [ ]:
censo.groupby("plataforma").agg(dominios=("id", "count"), online=("online", lambda s: (s == "OK").mean())).sort_values("dominios", ascending=False).head(12).round(3)